# Pasos previos

## Imports y dependencias

In [1]:
pip install scikit-learn pandas numpy gensim nltk matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
import re
from nltk.tokenize import TweetTokenizer, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import GaussianNB, MultinomialNB
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import numpy as np
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, v_measure_score
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans
# Importamos las métricas necesarias de scikit-learn
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.metrics.cluster import contingency_matrix


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/agustinramostorres/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/agustinramostorres/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Carga de archivo y preprocesamiento

In [3]:
tknzr =  TweetTokenizer(preserve_case=False)
stemmer = SnowballStemmer('spanish')

nltk.download("stopwords", quiet=True)
stop_es = set(stopwords.words("spanish"))

def preprocesar(text): # Preprocesamiento hecho
  texto = text.strip()
  texto = tknzr.tokenize(texto)
  texto = [stemmer.stem(t) for t in texto if t not in stop_es]
  texto = ' '.join(texto)
  patron = r'[^\w\sáéíóúüñáàèìòùäëïöü]'
  texto = re.sub(patron, '', texto)
  texto = ' '.join(texto.split())
  return texto

In [4]:
name = 'news_reducido.csv'

# Leer los datos en formato csv
data = pd.read_csv(name, on_bad_lines='skip')

# Nos quedamos con el texto (puedes quedarte con más información si quieres)
X = np.array([preprocesar(t) for t in data['text'].fillna('')])

## Datos utilizados

In [5]:
semilla1=42
semilla2=640
semilla3=5300742

semilla = semilla2

#Agrupamiento
num_clusters=4

#Clasificación
pesos = ["uniform", "distance"]
valor_p = [1, 2]
valor_k = [3, 4, 5, 6, 7, 10]

#Clasificación
peso="uniform"
p=1
k=3

# Definimos la función para calcular la Pureza (Purity)
def purity_score(y_true, y_pred):
    # Generamos la matriz de contingencia
    c_matrix = contingency_matrix(y_true, y_pred)
    # Calculamos la pureza
    return np.sum(np.amax(c_matrix, axis=0)) / np.sum(c_matrix)


## Validación cruzada estratificada

In [6]:
enc = OrdinalEncoder()
y = enc.fit_transform(np.reshape(data['category'], (-1,1))).reshape(-1) # DOCUMENTOS Y LE ASOCIO LA CATEGORIA

# Hacemos la partición train-test con Validacion cruzada estratificada
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf.get_n_splits(X, y)

5

# Agrupamiento

## Definición de los modelos

### Representación binaria

In [7]:
text_kmeans_binario = Pipeline([
    ('vect', CountVectorizer(binary=True)),
    ('clf', KMeans(n_clusters=num_clusters, random_state=semilla))
])

### Representación de frecuencias

In [8]:
text_kmeans_tf = Pipeline([
    ('vect', CountVectorizer(binary=False)),
    ('clf', KMeans(n_clusters=num_clusters, random_state=semilla))
])

### Representación TF-IDF

In [9]:
text_kmeans_tfidf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', KMeans(n_clusters=num_clusters, random_state=semilla))
])

### Algoritmo de mezcla gaussiana

In [10]:
text_kmeans_tfidf_mezcla_gausiana = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('to_dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
    ('clf', GaussianMixture(n_components=num_clusters, random_state=semilla, covariance_type='spherical'))
])

## Ejecución de los modelos

### Representación binaria

In [11]:
text_kmeans = text_kmeans_binario

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas

    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

Fold 1: 0.4515
Fold 2: 0.139
Fold 3: 0.3
Fold 4: 0.438
Fold 5: 0.356
Precision media = 0.3369


In [12]:
text_kmeans = text_kmeans_binario

accuracies = np.zeros(5)
silhouettes = np.zeros(5)
davies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X, y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])

    # Test (Predicción de etiquetas)
    labels = text_kmeans.predict(X[tst])

    # Transformamos el texto de test a su representación numérica (matriz)
    # Necesario porque las métricas matemáticas evalúan distancias entre puntos
    X_tst_vect = text_kmeans.named_steps['vect'].transform(X[tst])

    # Convertimos la matriz dispersa a densa para Calinski-Harabasz y Davies-Bouldin
    X_tst_dense = X_tst_vect.toarray()

    # Cálculo de métricas
    acc = np.mean(labels == y[tst])
    sil_score = silhouette_score(X_tst_dense, labels)
    db_score = davies_bouldin_score(X_tst_dense, labels)

    print(f"Fold {i+1} - Precision: {acc:} | Silhouette: {sil_score} | Davies-B: {db_score}")

    # Almacenamos los resultados
    accuracies[i] = acc
    silhouettes[i] = sil_score
    davies[i] = db_score

# Tras el K-Fold, mostramos las métricas medias
print("\n--- Resultados Promedio ---")
print(f"Precision media = {np.average(accuracies)}")
print(f"Silhouette media = {np.average(silhouettes)}")
print(f"Davies-Bouldin media = {np.average(davies)}")

Fold 1 - Precision: 0.4515 | Silhouette: 0.07524254316418807 | Davies-B: 6.398816264989281
Fold 2 - Precision: 0.139 | Silhouette: 0.036554723190414744 | Davies-B: 6.977817793350256
Fold 3 - Precision: 0.3 | Silhouette: 0.10174684636492802 | Davies-B: 5.4599139931570795
Fold 4 - Precision: 0.438 | Silhouette: 0.08826575303876333 | Davies-B: 6.125498397900465
Fold 5 - Precision: 0.356 | Silhouette: 0.10456673315405193 | Davies-B: 5.411236613727134

--- Resultados Promedio ---
Precision media = 0.3369
Silhouette media = 0.08127531978246923
Davies-Bouldin media = 6.074656612624843


In [13]:
text_kmeans = text_kmeans_binario

purities = np.zeros(5)
nmis = np.zeros(5)
aris = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X, y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])

    # Predicción (Test)
    labels = text_kmeans.predict(X[tst])

    # Cálculo de métricas
    fold_purity = purity_score(y[tst], labels)
    fold_nmi = normalized_mutual_info_score(y[tst], labels)
    fold_ari = adjusted_rand_score(y[tst], labels)

    print(f"--- Fold {i+1} ---")
    print(f"Pureza: {fold_purity} | NMI: {fold_nmi} | ARI: {fold_ari}")

    # Guardamos los resultados
    purities[i] = fold_purity
    nmis[i] = fold_nmi
    aris[i] = fold_ari

print("\n=======================================")
print("RESULTADOS GLOBALES (Medias tras 5 Folds)")
print("=======================================")
print(f"Pureza media = {np.mean(purities)}")
print(f"NMI medio    = {np.mean(nmis)}")
print(f"ARI medio    = {np.mean(aris)}")

--- Fold 1 ---
Pureza: 0.485 | NMI: 0.2338425469051787 | ARI: 0.14405680673974747
--- Fold 2 ---
Pureza: 0.5165 | NMI: 0.24700133723789142 | ARI: 0.1716869514566502
--- Fold 3 ---
Pureza: 0.3835 | NMI: 0.08954700429552309 | ARI: 0.07842075658611201
--- Fold 4 ---
Pureza: 0.4925 | NMI: 0.2571282586181214 | ARI: 0.15052686144513985
--- Fold 5 ---
Pureza: 0.3835 | NMI: 0.09564345807545184 | ARI: 0.0847791585423911

RESULTADOS GLOBALES (Medias tras 5 Folds)
Pureza media = 0.45220000000000005
NMI medio    = 0.1846325210264333
ARI medio    = 0.12589410695400813


### Representación de frecuencias

In [14]:
text_kmeans = text_kmeans_tf

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas
    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

Fold 1: 0.3125
Fold 2: 0.38
Fold 3: 0.2325
Fold 4: 0.3265
Fold 5: 0.303
Precision media = 0.3109


In [15]:
text_kmeans = text_kmeans_tf

accuracies = np.zeros(5)
silhouettes = np.zeros(5)
davies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X, y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])

    # Test (Predicción de etiquetas)
    labels = text_kmeans.predict(X[tst])

    # Transformamos el texto de test a su representación numérica (matriz)
    # Necesario porque las métricas matemáticas evalúan distancias entre puntos
    X_tst_vect = text_kmeans.named_steps['vect'].transform(X[tst])

    # Convertimos la matriz dispersa a densa para Calinski-Harabasz y Davies-Bouldin
    X_tst_dense = X_tst_vect.toarray()

    # Cálculo de métricas
    acc = np.mean(labels == y[tst])
    sil_score = silhouette_score(X_tst_dense, labels)
    db_score = davies_bouldin_score(X_tst_dense, labels)

    print(f"Fold {i+1} - Precision: {acc:} | Silhouette: {sil_score} | Davies-B: {db_score}")

    # Almacenamos los resultados
    accuracies[i] = acc
    silhouettes[i] = sil_score
    davies[i] = db_score

# Tras el K-Fold, mostramos las métricas medias
print("\n--- Resultados Promedio ---")
print(f"Precision media = {np.average(accuracies)}")
print(f"Silhouette media = {np.average(silhouettes)}")
print(f"Davies-Bouldin media = {np.average(davies)}")

Fold 1 - Precision: 0.3125 | Silhouette: 0.2549137592403752 | Davies-B: 1.7775368861985812
Fold 2 - Precision: 0.38 | Silhouette: 0.2758820303641156 | Davies-B: 1.7698710481185933
Fold 3 - Precision: 0.2325 | Silhouette: 0.2622947134001989 | Davies-B: 1.7321325376744312
Fold 4 - Precision: 0.3265 | Silhouette: 0.26867230113829504 | Davies-B: 1.7614362718359937
Fold 5 - Precision: 0.303 | Silhouette: 0.2767507707052524 | Davies-B: 1.7705001386124852

--- Resultados Promedio ---
Precision media = 0.3109
Silhouette media = 0.2677027149696474
Davies-Bouldin media = 1.762295376488017


In [16]:
text_kmeans = text_kmeans_tf

purities = np.zeros(5)
nmis = np.zeros(5)
aris = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X, y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])

    # Predicción (Test)
    labels = text_kmeans.predict(X[tst])

    # Cálculo de métricas
    fold_purity = purity_score(y[tst], labels)
    fold_nmi = normalized_mutual_info_score(y[tst], labels)
    fold_ari = adjusted_rand_score(y[tst], labels)

    print(f"--- Fold {i+1} ---")
    print(f"Pureza: {fold_purity} | NMI: {fold_nmi} | ARI: {fold_ari}")

    # Guardamos los resultados
    purities[i] = fold_purity
    nmis[i] = fold_nmi
    aris[i] = fold_ari

print("\n=======================================")
print("RESULTADOS GLOBALES (Medias tras 5 Folds)")
print("=======================================")
print(f"Pureza media = {np.mean(purities)}")
print(f"NMI medio    = {np.mean(nmis)}")
print(f"ARI medio    = {np.mean(aris)}")

--- Fold 1 ---
Pureza: 0.377 | NMI: 0.06670943510437553 | ARI: 0.05902855916354989
--- Fold 2 ---
Pureza: 0.38 | NMI: 0.06408571743736866 | ARI: 0.05603743860930571
--- Fold 3 ---
Pureza: 0.3765 | NMI: 0.05975113583935087 | ARI: 0.0522173296633404
--- Fold 4 ---
Pureza: 0.38 | NMI: 0.06736503334620271 | ARI: 0.058148422137268195
--- Fold 5 ---
Pureza: 0.391 | NMI: 0.07178059062566221 | ARI: 0.06411035793894591

RESULTADOS GLOBALES (Medias tras 5 Folds)
Pureza media = 0.3809
NMI medio    = 0.065938382470592
ARI medio    = 0.05790842150248202


### Representación TF-IDF

In [17]:
text_kmeans = text_kmeans_tfidf

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas
    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

Fold 1: 0.4915
Fold 2: 0.513
Fold 3: 0.1485
Fold 4: 0.4505
Fold 5: 0.1515
Precision media = 0.351


In [18]:
text_kmeans = text_kmeans_tfidf

accuracies = np.zeros(5)
silhouettes = np.zeros(5)
davies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X, y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])

    # Test (Predicción de etiquetas)
    labels = text_kmeans.predict(X[tst])

    # Transformamos el texto de test a su representación numérica (matriz)
    # Necesario porque las métricas matemáticas evalúan distancias entre puntos
    X_tst_vect = text_kmeans.named_steps['vect'].transform(X[tst])

    # Convertimos la matriz dispersa a densa para Calinski-Harabasz y Davies-Bouldin
    X_tst_dense = X_tst_vect.toarray()

    # Cálculo de métricas
    acc = np.mean(labels == y[tst])
    sil_score = silhouette_score(X_tst_dense, labels)
    db_score = davies_bouldin_score(X_tst_dense, labels)

    print(f"Fold {i+1} - Precision: {acc:} | Silhouette: {sil_score} | Davies-B: {db_score}")

    # Almacenamos los resultados
    accuracies[i] = acc
    silhouettes[i] = sil_score
    davies[i] = db_score

# Tras el K-Fold, mostramos las métricas medias
print("\n--- Resultados Promedio ---")
print(f"Precision media = {np.average(accuracies)}")
print(f"Silhouette media = {np.average(silhouettes)}")
print(f"Davies-Bouldin media = {np.average(davies)}")

Fold 1 - Precision: 0.4915 | Silhouette: -0.20055529515032167 | Davies-B: 2.767371410712879
Fold 2 - Precision: 0.513 | Silhouette: 0.07654991585783333 | Davies-B: 3.06382311263252
Fold 3 - Precision: 0.1485 | Silhouette: -0.1882146413740852 | Davies-B: 2.9045077183445467
Fold 4 - Precision: 0.4505 | Silhouette: 0.0679936096434598 | Davies-B: 4.15937348707775
Fold 5 - Precision: 0.1515 | Silhouette: 0.05529268658204338 | Davies-B: 3.26746623829407

--- Resultados Promedio ---
Precision media = 0.351
Silhouette media = -0.03778674488821406
Davies-Bouldin media = 3.2325083934123535


In [19]:
text_kmeans = text_kmeans_tfidf

purities = np.zeros(5)
nmis = np.zeros(5)
aris = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X, y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])

    # Predicción (Test)
    labels = text_kmeans.predict(X[tst])

    # Cálculo de métricas
    fold_purity = purity_score(y[tst], labels)
    fold_nmi = normalized_mutual_info_score(y[tst], labels)
    fold_ari = adjusted_rand_score(y[tst], labels)

    print(f"--- Fold {i+1} ---")
    print(f"Pureza: {fold_purity} | NMI: {fold_nmi} | ARI: {fold_ari}")

    # Guardamos los resultados
    purities[i] = fold_purity
    nmis[i] = fold_nmi
    aris[i] = fold_ari

print("\n=======================================")
print("RESULTADOS GLOBALES (Medias tras 5 Folds)")
print("=======================================")
print(f"Pureza media = {np.mean(purities)}")
print(f"NMI medio    = {np.mean(nmis)}")
print(f"ARI medio    = {np.mean(aris)}")

--- Fold 1 ---
Pureza: 0.5745 | NMI: 0.3419579899202876 | ARI: 0.27193008584856987
--- Fold 2 ---
Pureza: 0.5875 | NMI: 0.36859466328862633 | ARI: 0.2997830062358806
--- Fold 3 ---
Pureza: 0.5915 | NMI: 0.3731178549639298 | ARI: 0.3063605660485983
--- Fold 4 ---
Pureza: 0.621 | NMI: 0.34674556758652003 | ARI: 0.2862947940376696
--- Fold 5 ---
Pureza: 0.5815 | NMI: 0.36592289635939 | ARI: 0.28710966457543863

RESULTADOS GLOBALES (Medias tras 5 Folds)
Pureza media = 0.5912
NMI medio    = 0.3592677944237508
ARI medio    = 0.2902956233492314


### Algoritmo de mezcla gaussiana


In [ ]:
text_kmeans = text_kmeans_tfidf_mezcla_gausiana

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas
    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interés, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

## Representación del mejor y el peor fold con el algoritmo T-SNE

### Peor fold: Representación: TF-IDF, Algoritmo: K-Means, Semilla: 42, Fold: 5

In [ ]:
# Este te saca las cosas con las fotos
text_kmeans = text_kmeans_tfidf

folds = list(skf.split(X,y))
tra, tst = folds[4]

# Entrenamiento
text_kmeans.fit(X[tra])
# Test
labels = text_kmeans.predict(X[tst])
# Calculo de metricas
acc = np.mean(labels == y[tst])
print(acc)

X_tst_transformed = text_kmeans[:-1].transform(X[tst])

tsne = TSNE(
    n_components=2,
    # Aseguramos que la perplejidad no sea mayor que el número de muestras
    perplexity=min(30, max(5, X_tst_transformed.shape[0] // 3)),
    learning_rate='auto',
    init='pca',
    random_state=42
)

tst_2d = tsne.fit_transform(X_tst_transformed)

plt.figure(figsize=(9, 6))
# Usamos 'labels' que son las predicciones que hiciste arriba
plt.scatter(tst_2d[:, 0], tst_2d[:, 1], c=labels, s=12, cmap='tab10', alpha=0.8)
plt.title(f"t-SNE fold {5} (Representación TF-IDF)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(label="Cluster Predicho")
plt.tight_layout()
nombre_archivo = "./tsne_fold_5_clusters_tst.png"
plt.savefig(nombre_archivo, dpi=300, bbox_inches="tight")
print(f"Imagen guardada con éxito como: {nombre_archivo}")
plt.close()


# 1. Creamos un DataFrame con los nombres reales de las noticias del test
# y los IDs que el KMeans se ha inventado (labels)
df_mapping = pd.DataFrame({
    'KMeans_Cluster': labels,
    'Nombre_Real': data.loc[tst]['category'].values
})

# 2. Generamos la tabla de frecuencias
# Esto nos dirá cuántas noticias de cada categoría hay en cada clúster
tabla_frecuencias = pd.crosstab(df_mapping['KMeans_Cluster'], df_mapping['Nombre_Real'])

print("--- Tabla de Correspondencia ---")
print(tabla_frecuencias)

# 3. Código para decirte automáticamente cuál es cuál
print("\n--- Interpretación sugerida ---")
for cluster_id in tabla_frecuencias.index:
    # Buscamos la categoría con el valor máximo en esa fila
    categoria_predominante = tabla_frecuencias.columns[tabla_frecuencias.loc[cluster_id].argmax()]
    total_muestras = tabla_frecuencias.loc[cluster_id].sum()
    print(f"Clúster {cluster_id}: Probablemente es '{categoria_predominante}' ({total_muestras} noticias)")

### Mejor fold: Representación: TF-IDF, Algoritmo: K-Means, Semilla: 640, Fold: 2

In [ ]:
# Este te saca las cosas con las fotos
text_kmeans = text_kmeans_tfidf

folds = list(skf.split(X,y))
# Cambiado a la tercera posición (índice 2)
tra, tst = folds[1]

# Entrenamiento
text_kmeans.fit(X[tra])
# Test
labels = text_kmeans.predict(X[tst])
# Calculo de metricas
acc = np.mean(labels == y[tst])
print(acc)

print("Valores únicos en labels:", np.unique(labels))
print("Frecuencia por clúster:", np.bincount(labels))

X_tst_transformed = text_kmeans[:-1].transform(X[tst])

tsne = TSNE(
    n_components=2,
    # Aseguramos que la perplejidad no sea mayor que el número de muestras
    perplexity=min(30, max(5, X_tst_transformed.shape[0] // 3)),
    learning_rate='auto',
    init='pca',
    random_state=42
)

tst_2d = tsne.fit_transform(X_tst_transformed)

plt.figure(figsize=(9, 6))
# Usamos 'labels' que son las predicciones que hiciste arriba
plt.scatter(tst_2d[:, 0], tst_2d[:, 1], c=labels, s=12, cmap='tab10', alpha=0.8)
# Actualizado el título y el nombre del archivo al Fold 3
plt.title(f"t-SNE fold {2} (Representación TF-IDF)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(label="Cluster Predicho")
plt.tight_layout()
nombre_archivo = "./tsne_fold_2_clusters_tst.png"
plt.savefig(nombre_archivo, dpi=300, bbox_inches="tight")
print(f"Imagen guardada con éxito como: {nombre_archivo}")
plt.close()


# 1. Creamos un DataFrame con los nombres reales de las noticias del test
# y los IDs que el KMeans se ha inventado (labels)
df_mapping = pd.DataFrame({
    'KMeans_Cluster': labels,
    'Nombre_Real': data.loc[tst]['category'].values
})

# 2. Generamos la tabla de frecuencias
# Esto nos dirá cuántas noticias de cada categoría hay en cada clúster
tabla_frecuencias = pd.crosstab(df_mapping['KMeans_Cluster'], df_mapping['Nombre_Real'])

print("--- Tabla de Correspondencia ---")
print(tabla_frecuencias)

# 3. Código para decirte automáticamente cuál es cuál
print("\n--- Interpretación sugerida ---")
for cluster_id in tabla_frecuencias.index:
    # Buscamos la categoría con el valor máximo en esa fila
    categoria_predominante = tabla_frecuencias.columns[tabla_frecuencias.loc[cluster_id].argmax()]
    total_muestras = tabla_frecuencias.loc[cluster_id].sum()
    print(f"Clúster {cluster_id}: Probablemente es '{categoria_predominante}' ({total_muestras} noticias)")

### Mejor combinación binaria: Algoritmo: K-Means, Semilla: 640, Fold: 1

In [ ]:
# Este te saca las cosas con las fotos
text_kmeans = text_kmeans_binario

folds = list(skf.split(X,y))
tra, tst = folds[0]

# Entrenamiento
text_kmeans.fit(X[tra])
# Test
labels = text_kmeans.predict(X[tst])
# Calculo de metricas
acc = np.mean(labels == y[tst])
print(acc)

X_tst_transformed = text_kmeans[:-1].transform(X[tst])

tsne = TSNE(
    n_components=2,
    # Aseguramos que la perplejidad no sea mayor que el número de muestras
    perplexity=min(30, max(5, X_tst_transformed.shape[0] // 3)),
    learning_rate='auto',
    init='pca',
    random_state=42
)

tst_2d = tsne.fit_transform(X_tst_transformed)

plt.figure(figsize=(9, 6))
# Usamos 'labels' que son las predicciones que hiciste arriba
plt.scatter(tst_2d[:, 0], tst_2d[:, 1], c=labels, s=12, cmap='tab10', alpha=0.8)
plt.title(f"t-SNE fold {1} (Representación Binaria)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(label="Cluster Predicho")
plt.tight_layout()
nombre_archivo = "./tsne_fold_1_clusters_tst.png"
plt.savefig(nombre_archivo, dpi=300, bbox_inches="tight")
print(f"Imagen guardada con éxito como: {nombre_archivo}")
plt.close()

### Mejor combinación frecuencias: Algoritmo: K-Means, Semilla: 640, Fold: 2

In [ ]:
# Este te saca las cosas con las fotos
text_kmeans = text_kmeans_tf

folds = list(skf.split(X, y))
tra, tst = folds[1]

# Entrenamiento
text_kmeans.fit(X[tra])
# Test
labels = text_kmeans.predict(X[tst])
# Calculo de metricas
acc = np.mean(labels == y[tst])
print(acc)

X_tst_transformed = text_kmeans[:-1].transform(X[tst])

tsne = TSNE(
    n_components=2,
    # Aseguramos que la perplejidad no sea mayor que el número de muestras
    perplexity=min(30, max(5, X_tst_transformed.shape[0] // 3)),
    learning_rate='auto',
    init='pca',
    random_state=42
)

tst_2d = tsne.fit_transform(X_tst_transformed)

plt.figure(figsize=(9, 6))
# Usamos 'labels' que son las predicciones que hiciste arriba
plt.scatter(tst_2d[:, 0], tst_2d[:, 1], c=labels, s=12, cmap='tab10', alpha=0.8)
plt.title(f"t-SNE fold {2} (Representación frecuencias)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(label="Cluster Predicho")
plt.tight_layout()
nombre_archivo = "./tsne_fold_2_clusters_tst.png"
plt.savefig(nombre_archivo, dpi=300, bbox_inches="tight")
print(f"Imagen guardada con éxito como: {nombre_archivo}")
plt.close()

# Clasificación

## Definición de los modelos

### Binaria

In [ ]:
text_clasifier_binario = Pipeline([
    ('vect', CountVectorizer(binary=True)),
    ('clf', KNeighborsClassifier()) #Preguntar los randoms state
])

### Frecuencias

In [ ]:
text_clasifier_tf = Pipeline([
    ('vect', CountVectorizer(binary=False)),
    ('clf', KNeighborsClassifier()) #Preguntar el random_state
])

### TF-IDF

In [ ]:
text_clasifier_tfidf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', KNeighborsClassifier())
])

### Gausiana Binaria

In [ ]:
text_Gausian_binaria = Pipeline([
    ('vect', CountVectorizer(binary=True)),
    ('to_dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
    ('clf', GaussianNB())
])

### Gausiana Frecuencias

In [ ]:
text_Gausian_frecuencias = Pipeline([
    ('vect', CountVectorizer(binary=False)),
    ('to_dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
    ('clf', GaussianNB())
])

### Gausiana TF-IDF


In [ ]:
text_Gausian_tfidf = Pipeline([ # Solo para TF-IDF # No funciona en el servidor, se caga encima
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('to_dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
    ('clf', GaussianNB())
])

### Multinomial binario

In [ ]:
text_Multinomial_binario = Pipeline([
    ('vect', CountVectorizer(binary=True)),
    ('clf', MultinomialNB())
])

### Multinomial frecuencias

In [ ]:
text_Multinomial_frecuencias = Pipeline([
    ('vect', CountVectorizer(binary=False)),
    ('clf', MultinomialNB())
])

### Multinomial TF-IDF

In [ ]:
text_Multinomial_tfidf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB())
])

## Ejecución de los modelos

### Binaria

In [ ]:
text_sgd = text_clasifier_binario

text_sgd.set_params(
    clf__weights=peso,
    clf__p=p,
    clf__n_neighbors=k
)

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Frecuencias

In [ ]:
text_sgd = text_clasifier_tf

text_sgd.set_params(
    clf__weights=peso,
    clf__p=p,
    clf__n_neighbors=k
)

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### TF-IDF

In [ ]:
text_sgd = text_clasifier_tfidf

text_sgd.set_params(
    clf__weights=peso,
    clf__p=p,
    clf__n_neighbors=k
)

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Gausiana binaria

In [ ]:
text_sgd = text_Gausian_binaria

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Gausiana frecuencias

In [ ]:
text_sgd = text_Gausian_frecuencias

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Gausiana TF-IDF

In [ ]:
text_sgd = text_Gausian_tfidf

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Multinomial binario

In [ ]:
text_sgd = text_Multinomial_binario

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Multinomial frecuencias

In [ ]:
text_sgd = text_Multinomial_frecuencias

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])

    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Multinomial TF-IDF

In [ ]:
text_sgd = text_Multinomial_tfidf

accuracies = np.zeros(5)
for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamientoo
    text_sgd.fit(X[tra], y[tra])
    # Test (obtener predicciones)
    predicted = text_sgd.predict(X[tst])

    # Calculo de metricas de calidad (ahora, solo accuracy)
    acc = np.mean(predicted == y[tst])

    print(acc)
    accuracies[i] = acc
# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Script para la automatización de las pruebas de los agoritmos de K-NN

In [ ]:
pesos = ["uniform", "distance"]
valor_p = [1, 2]
valor_k = [3, 4, 5, 6, 7, 10]

results = []

pipelines = {
    'binary': text_clasifier_binario,
    'tf':     text_clasifier_tf,
    'tfidf':  text_clasifier_tfidf
}


for nombre_pipe, pipeline in pipelines.items():
    for peso in pesos:
        for p in valor_p:
            for k in valor_k:
                # Actualizar hiperparámetros del clasificador dentro del pipeline
                pipeline.set_params(
                    clf__weights=peso,
                    clf__p=p,
                    clf__n_neighbors=k
                )

                accuracies = np.zeros(5)
                for i, (tra, tst) in enumerate(skf.split(X, y)):
                    pipeline.fit(X[tra], y[tra])
                    predicted = pipeline.predict(X[tst])
                    accuracies[i] = np.mean(predicted == y[tst])
                    print("Fold " + str(i+1) +": " + str(accuracies[i]))

                avg_acc = np.average(accuracies)
                results.append({
                    'semilla':     semilla,
                    'vectorizer':  nombre_pipe,
                    'peso':        peso,
                    'p':           p,
                    'k':           k,
                    'avg_acc':     avg_acc
                })
                print(f'semilla={semilla} | vec={nombre_pipe} | peso={peso} | p={p} | k={k} → acc={avg_acc}')


In [ ]:
# Convertir a DataFrame para analizar mejor los resultados
results_df = pd.DataFrame(results)
top_3 = results_df.nlargest(3, 'avg_acc')

bottom_2 = results_df.nsmallest(2, 'avg_acc')

print("=== TOP 3 MEJORES CONFIGURACIONES ===")
print(top_3)

print("\n=== 2 PEORES CONFIGURACIONES ===")
print(bottom_2)